# Production Evaluation: Deploying LLM-as-a-Judge at Scale

This notebook covers production considerations for deploying LLM-as-a-Judge evaluation systems.

## The Production Challenge

Moving from development to production introduces new challenges:
- **Cost**: Evaluation at scale can be expensive
- **Latency**: Real-time evaluation must be fast
- **Reliability**: Systems must handle failures gracefully
- **Monitoring**: Track quality trends over time
- **Scalability**: Handle varying load efficiently

## What You'll Learn

1. Cost optimization strategies
2. Latency reduction techniques
3. Hybrid evaluation approaches
4. Monitoring and alerting
5. Best practices for production deployment
6. Decision frameworks for evaluation strategy

## Prerequisites

Complete the previous notebooks:
- [LLM as a Judge Basics](LLM_as_Judge_Basics.ipynb)
- [Ensemble Judges](Ensemble_Judges.ipynb)

# Part 1: Cost Optimization

## Understanding Evaluation Costs

LLM-as-a-Judge evaluation costs come from:
- **Input tokens**: User query + agent response + context
- **Output tokens**: Judge's evaluation response
- **Frequency**: How often you evaluate
- **Number of judges**: Single vs ensemble

### Cost Calculation Example

Assumptions:
- Average input: 500 tokens (query + response + context)
- Average output: 300 tokens (evaluation)
- Cost: $0.01 per 1K tokens (example rate)
- Traffic: 10,000 requests/day

**Single Judge:**
- Tokens per evaluation: 800
- Daily cost: 10,000 × 800 / 1000 × $0.01 = $80/day
- Monthly cost: ~$2,400

**Ensemble (3 judges):**
- Daily cost: $240/day
- Monthly cost: ~$7,200

## Cost Optimization Strategies

### Strategy 1: Sampling

Evaluate only a percentage of production traffic.

In [ ]:
import random
from typing import Optional

class SamplingEvaluator:
    """Evaluate only a sample of requests."""
    
    def __init__(self, sample_rate: float = 0.1, judge_model=None):
        """
        Args:
            sample_rate: Fraction of requests to evaluate (0.0-1.0)
            judge_model: The judge model to use
        """
        self.sample_rate = sample_rate
        self.judge_model = judge_model
        self.evaluated_count = 0
        self.total_count = 0
    
    def should_evaluate(self) -> bool:
        """Decide whether to evaluate this request."""
        return random.random() < self.sample_rate
    
    def evaluate(self, user_query: str, agent_response: str, 
                tool_results: Optional[str] = None) -> Optional[dict]:
        """Evaluate if sampled, otherwise return None."""
        self.total_count += 1
        
        if not self.should_evaluate():
            return None
        
        self.evaluated_count += 1
        # Actual evaluation would happen here
        return {
            "evaluated": True,
            "sample_rate": self.sample_rate,
            "evaluated_count": self.evaluated_count,
            "total_count": self.total_count
        }
    
    def get_stats(self) -> dict:
        """Get evaluation statistics."""
        return {
            "total_requests": self.total_count,
            "evaluated_requests": self.evaluated_count,
            "actual_rate": self.evaluated_count / max(1, self.total_count),
            "target_rate": self.sample_rate
        }

# Example usage
evaluator = SamplingEvaluator(sample_rate=0.1)  # Evaluate 10%

# Simulate 100 requests
for i in range(100):
    result = evaluator.evaluate(f"query_{i}", f"response_{i}")

stats = evaluator.get_stats()
print(f"\nSampling Statistics:")
print(f"  Total requests: {stats['total_requests']}")
print(f"  Evaluated: {stats['evaluated_requests']}")
print(f"  Actual rate: {stats['actual_rate']:.1%}")
print(f"  Cost reduction: {(1 - stats['actual_rate']):.1%}")

### Strategy 2: Caching

Cache evaluation results for identical inputs.

In [ ]:
import hashlib
from typing import Dict, Any

class CachingEvaluator:
    """Cache evaluation results to avoid redundant evaluations."""
    
    def __init__(self, judge_model=None, max_cache_size: int = 1000):
        self.judge_model = judge_model
        self.cache: Dict[str, Any] = {}
        self.max_cache_size = max_cache_size
        self.cache_hits = 0
        self.cache_misses = 0
    
    def _create_cache_key(self, user_query: str, agent_response: str, 
                         tool_results: Optional[str] = None) -> str:
        """Create a hash key for caching."""
        content = f"{user_query}|{agent_response}|{tool_results or ''}"
        return hashlib.md5(content.encode()).hexdigest()
    
    def evaluate(self, user_query: str, agent_response: str,
                tool_results: Optional[str] = None) -> dict:
        """Evaluate with caching."""
        cache_key = self._create_cache_key(user_query, agent_response, tool_results)
        
        # Check cache
        if cache_key in self.cache:
            self.cache_hits += 1
            return {**self.cache[cache_key], "from_cache": True}
        
        # Cache miss - evaluate
        self.cache_misses += 1
        result = {
            "score": 8.5,  # Placeholder
            "from_cache": False
        }
        
        # Store in cache (with size limit)
        if len(self.cache) < self.max_cache_size:
            self.cache[cache_key] = result
        
        return result
    
    def get_cache_stats(self) -> dict:
        """Get cache performance statistics."""
        total = self.cache_hits + self.cache_misses
        return {
            "cache_size": len(self.cache),
            "cache_hits": self.cache_hits,
            "cache_misses": self.cache_misses,
            "hit_rate": self.cache_hits / max(1, total),
            "cost_savings": self.cache_hits / max(1, total)
        }

# Example usage
evaluator = CachingEvaluator()

# Simulate requests with some duplicates
queries = ["weather in Miami", "stock price IBM", "weather in Miami", "weather in Miami"]
for q in queries:
    result = evaluator.evaluate(q, f"response for {q}")
    print(f"Query: {q[:20]}... | From cache: {result['from_cache']}")

stats = evaluator.get_cache_stats()
print(f"\nCache Statistics:")
print(f"  Hit rate: {stats['hit_rate']:.1%}")
print(f"  Cost savings: {stats['cost_savings']:.1%}")

### Strategy 3: Tiered Evaluation

Use fast checks first, expensive judges only when needed.

In [ ]:
class TieredEvaluator:
    """Multi-tier evaluation: fast checks first, LLM judge for edge cases."""
    
    def __init__(self, judge_model=None):
        self.judge_model = judge_model
        self.tier1_count = 0  # Fast checks
        self.tier2_count = 0  # LLM judge
    
    def tier1_check(self, user_query: str, agent_response: str) -> Optional[dict]:
        """Fast, cheap checks (substring matching, length, etc.)."""
        self.tier1_count += 1
        
        # Example: Check if response is too short
        if len(agent_response) < 10:
            return {"tier": 1, "pass": False, "reason": "Response too short"}
        
        # Example: Check for error messages
        if "error" in agent_response.lower():
            return {"tier": 1, "pass": False, "reason": "Contains error"}
        
        # Passed tier 1 - needs tier 2
        return None
    
    def tier2_evaluate(self, user_query: str, agent_response: str,
                      tool_results: Optional[str] = None) -> dict:
        """Expensive LLM-based evaluation."""
        self.tier2_count += 1
        # Actual LLM evaluation would happen here
        return {"tier": 2, "pass": True, "score": 8.5}
    
    def evaluate(self, user_query: str, agent_response: str,
                tool_results: Optional[str] = None) -> dict:
        """Tiered evaluation."""
        # Try tier 1 first
        tier1_result = self.tier1_check(user_query, agent_response)
        if tier1_result is not None:
            return tier1_result
        
        # Fall back to tier 2
        return self.tier2_evaluate(user_query, agent_response, tool_results)
    
    def get_stats(self) -> dict:
        """Get tier usage statistics."""
        total = self.tier1_count
        return {
            "tier1_checks": self.tier1_count,
            "tier2_evaluations": self.tier2_count,
            "tier2_rate": self.tier2_count / max(1, total),
            "cost_savings": 1 - (self.tier2_count / max(1, total))
        }

# Example usage
evaluator = TieredEvaluator()

test_cases = [
    ("weather", "short"),  # Fails tier 1
    ("weather", "Error: API failed"),  # Fails tier 1
    ("weather", "The weather in Miami is partly cloudy"),  # Passes to tier 2
    ("stock", "IBM stock price is $245.45"),  # Passes to tier 2
]

for query, response in test_cases:
    result = evaluator.evaluate(query, response)
    print(f"Tier {result['tier']}: {response[:30]}... | Pass: {result.get('pass', 'N/A')}")

stats = evaluator.get_stats()
print(f"\nTiered Evaluation Stats:")
print(f"  Tier 2 rate: {stats['tier2_rate']:.1%}")
print(f"  Cost savings: {stats['cost_savings']:.1%}")

# Part 2: Latency Optimization

## Asynchronous Evaluation

Don't block user responses waiting for evaluation.

In [ ]:
import asyncio
from queue import Queue
from threading import Thread
import time

class AsyncEvaluator:
    """Evaluate asynchronously without blocking responses."""
    
    def __init__(self, judge_model=None, queue_size: int = 1000):
        self.judge_model = judge_model
        self.eval_queue = Queue(maxsize=queue_size)
        self.results = {}
        self.worker_thread = None
        self.running = False
    
    def start(self):
        """Start the background evaluation worker."""
        self.running = True
        self.worker_thread = Thread(target=self._worker, daemon=True)
        self.worker_thread.start()
    
    def stop(self):
        """Stop the background worker."""
        self.running = False
        if self.worker_thread:
            self.worker_thread.join(timeout=5)
    
    def _worker(self):
        """Background worker that processes evaluation queue."""
        while self.running:
            try:
                if not self.eval_queue.empty():
                    eval_id, user_query, agent_response, tool_results = self.eval_queue.get(timeout=1)
                    
                    # Simulate evaluation
                    time.sleep(0.1)  # Simulated LLM call
                    result = {"score": 8.5, "evaluated_at": time.time()}
                    
                    self.results[eval_id] = result
                    self.eval_queue.task_done()
                else:
                    time.sleep(0.1)
            except:
                continue
    
    def queue_evaluation(self, eval_id: str, user_query: str, 
                        agent_response: str, tool_results: Optional[str] = None):
        """Queue an evaluation (non-blocking)."""
        try:
            self.eval_queue.put_nowait((eval_id, user_query, agent_response, tool_results))
            return {"queued": True, "eval_id": eval_id}
        except:
            return {"queued": False, "reason": "Queue full"}
    
    def get_result(self, eval_id: str) -> Optional[dict]:
        """Get evaluation result if ready."""
        return self.results.get(eval_id)
    
    def get_stats(self) -> dict:
        """Get queue statistics."""
        return {
            "queue_size": self.eval_queue.qsize(),
            "completed_evaluations": len(self.results),
            "worker_running": self.running
        }

# Example usage
evaluator = AsyncEvaluator()
evaluator.start()

# Queue evaluations (non-blocking)
for i in range(5):
    result = evaluator.queue_evaluation(f"eval_{i}", f"query_{i}", f"response_{i}")
    print(f"Queued eval_{i}: {result['queued']}")

# User gets response immediately, evaluation happens in background
print("\nUser receives response immediately!")

# Check results later
time.sleep(1)
for i in range(5):
    result = evaluator.get_result(f"eval_{i}")
    print(f"eval_{i} result: {result}")

stats = evaluator.get_stats()
print(f"\nAsync Stats: {stats}")

evaluator.stop()

# Part 3: Monitoring and Alerting

Track evaluation metrics over time to detect quality issues.

In [ ]:
from collections import deque
from datetime import datetime
import statistics

class EvaluationMonitor:
    """Monitor evaluation metrics and trigger alerts."""
    
    def __init__(self, window_size: int = 100, alert_threshold: float = 6.0):
        self.window_size = window_size
        self.alert_threshold = alert_threshold
        self.scores = deque(maxlen=window_size)
        self.timestamps = deque(maxlen=window_size)
        self.alerts = []
    
    def record_evaluation(self, score: float):
        """Record an evaluation score."""
        self.scores.append(score)
        self.timestamps.append(datetime.now())
        
        # Check for alerts
        if len(self.scores) >= 10:  # Need minimum data
            recent_avg = statistics.mean(list(self.scores)[-10:])
            if recent_avg < self.alert_threshold:
                self.alerts.append({
                    "timestamp": datetime.now(),
                    "type": "low_quality",
                    "recent_avg": recent_avg,
                    "threshold": self.alert_threshold
                })
    
    def get_metrics(self) -> dict:
        """Get current metrics."""
        if not self.scores:
            return {"error": "No data"}
        
        scores_list = list(self.scores)
        return {
            "count": len(scores_list),
            "mean": statistics.mean(scores_list),
            "median": statistics.median(scores_list),
            "stdev": statistics.stdev(scores_list) if len(scores_list) > 1 else 0,
            "min": min(scores_list),
            "max": max(scores_list),
            "recent_10_avg": statistics.mean(scores_list[-10:]) if len(scores_list) >= 10 else None
        }
    
    def get_alerts(self) -> list:
        """Get recent alerts."""
        return self.alerts[-10:]  # Last 10 alerts
    
    def clear_alerts(self):
        """Clear alert history."""
        self.alerts.clear()

# Example usage
monitor = EvaluationMonitor(window_size=50, alert_threshold=6.5)

# Simulate evaluations with quality degradation
import random
for i in range(60):
    # Simulate quality drop after 40 evaluations
    if i < 40:
        score = random.uniform(7.5, 9.5)
    else:
        score = random.uniform(4.0, 6.0)  # Quality drops
    
    monitor.record_evaluation(score)

# Check metrics
metrics = monitor.get_metrics()
print("\nEvaluation Metrics:")
print(f"  Mean score: {metrics['mean']:.2f}")
print(f"  Median score: {metrics['median']:.2f}")
print(f"  Std dev: {metrics['stdev']:.2f}")
print(f"  Recent 10 avg: {metrics['recent_10_avg']:.2f}")

# Check alerts
alerts = monitor.get_alerts()
print(f"\n🚨 Alerts triggered: {len(alerts)}")
if alerts:
    latest = alerts[-1]
    print(f"  Latest: Recent avg {latest['recent_avg']:.2f} below threshold {latest['threshold']:.2f}")

# Part 4: Decision Framework

## When to Use Each Evaluation Approach

### Decision Tree

In [ ]:
def recommend_evaluation_strategy(
    traffic_volume: str,  # "low", "medium", "high"
    criticality: str,  # "low", "medium", "high"
    budget: str,  # "tight", "moderate", "flexible"
    latency_requirement: str  # "strict", "moderate", "flexible"
) -> dict:
    """
    Recommend evaluation strategy based on requirements.
    """
    recommendations = []
    
    # High criticality always needs robust evaluation
    if criticality == "high":
        recommendations.append("Use ensemble judges for reliability")
        recommendations.append("Consider human review for edge cases")
    
    # Budget constraints
    if budget == "tight":
        recommendations.append("Use sampling (10-20% of traffic)")
        recommendations.append("Implement caching for common queries")
        recommendations.append("Use tiered evaluation (fast checks first)")
    
    # Latency requirements
    if latency_requirement == "strict":
        recommendations.append("Use asynchronous evaluation")
        recommendations.append("Don't block user responses")
        recommendations.append("Consider programmatic checks for real-time validation")
    
    # Traffic volume
    if traffic_volume == "high":
        recommendations.append("Implement aggressive caching")
        recommendations.append("Use sampling to control costs")
        recommendations.append("Deploy async evaluation infrastructure")
    
    # Determine primary strategy
    if criticality == "high" and budget == "flexible":
        primary = "Ensemble judges with full coverage"
    elif criticality == "high" and budget == "tight":
        primary = "Single judge with sampling + human review for failures"
    elif criticality == "low" and budget == "tight":
        primary = "Programmatic checks + sampled LLM judge"
    else:
        primary = "Single LLM judge with caching"
    
    return {
        "primary_strategy": primary,
        "recommendations": recommendations,
        "estimated_cost_level": _estimate_cost(traffic_volume, criticality, budget)
    }

def _estimate_cost(traffic, criticality, budget):
    """Estimate relative cost level."""
    cost_map = {
        ("high", "high", "flexible"): "Very High",
        ("high", "high", "tight"): "High",
        ("medium", "medium", "moderate"): "Medium",
        ("low", "low", "tight"): "Low"
    }
    return cost_map.get((traffic, criticality, budget), "Medium")

# Example scenarios
scenarios = [
    {
        "name": "High-stakes financial agent",
        "traffic": "medium",
        "criticality": "high",
        "budget": "flexible",
        "latency": "moderate"
    },
    {
        "name": "Customer support chatbot",
        "traffic": "high",
        "criticality": "medium",
        "budget": "moderate",
        "latency": "strict"
    },
    {
        "name": "Internal tool assistant",
        "traffic": "low",
        "criticality": "low",
        "budget": "tight",
        "latency": "flexible"
    }
]

for scenario in scenarios:
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario['name']}")
    print(f"{'='*60}")
    
    recommendation = recommend_evaluation_strategy(
        scenario['traffic'],
        scenario['criticality'],
        scenario['budget'],
        scenario['latency']
    )
    
    print(f"\nPrimary Strategy: {recommendation['primary_strategy']}")
    print(f"Estimated Cost: {recommendation['estimated_cost_level']}")
    print(f"\nRecommendations:")
    for rec in recommendation['recommendations']:
        print(f"  • {rec}")

# Part 5: Best Practices Summary

## Cost Optimization

✅ **Do:**
- Use sampling for high-volume, low-stakes scenarios
- Implement caching for repeated queries
- Use tiered evaluation (fast checks first)
- Monitor cost trends and adjust strategies

❌ **Don't:**
- Evaluate 100% of traffic without justification
- Use ensemble judges when single judge suffices
- Ignore caching opportunities

## Latency Optimization

✅ **Do:**
- Use asynchronous evaluation
- Never block user responses for evaluation
- Batch evaluations when possible
- Use parallel execution for ensemble judges

❌ **Don't:**
- Make users wait for evaluation results
- Run evaluations synchronously in request path

## Monitoring

✅ **Do:**
- Track evaluation score distributions
- Set up alerts for quality degradation
- Monitor judge agreement metrics
- Correlate with user feedback

❌ **Don't:**
- Deploy without monitoring
- Ignore sudden score changes
- Rely solely on automated evaluation

## Hybrid Approaches

✅ **Do:**
- Combine programmatic checks with LLM judges
- Use different strategies for different use cases
- Escalate to human review when needed

❌ **Don't:**
- Use only one evaluation method
- Treat all requests identically

# Conclusion

## Key Takeaways

1. ✅ **Cost optimization** is critical for production deployment
2. ✅ **Asynchronous evaluation** prevents latency issues
3. ✅ **Monitoring** enables early detection of quality issues
4. ✅ **Hybrid approaches** balance cost, latency, and accuracy
5. ✅ **Decision frameworks** help choose the right strategy

## Production Checklist

Before deploying LLM-as-a-Judge evaluation:

- [ ] Estimate costs for expected traffic
- [ ] Implement sampling or caching strategy
- [ ] Set up asynchronous evaluation
- [ ] Configure monitoring and alerts
- [ ] Define escalation procedures
- [ ] Test with production-like load
- [ ] Document evaluation strategy
- [ ] Train team on interpreting metrics

## Comparison: Evaluation Approaches

| Approach | Cost | Latency | Accuracy | Best For |
|----------|------|---------|----------|----------|
| **Programmatic** | Free | <1ms | Limited | Tool call validation |
| **Single Judge** | $ | 100-500ms | Good | General evaluation |
| **Ensemble** | $$$ | 200-1000ms | Excellent | High-stakes decisions |
| **Sampled Judge** | $ | Async | Good | High-volume scenarios |
| **Tiered** | $-$$ | 1-500ms | Good | Cost-conscious deployments |
| **Human Review** | $$$$ | Hours | Excellent | Edge cases, calibration |

## Next Steps

1. **Review the full series:**
   - [LLM as a Judge Basics](LLM_as_Judge_Basics.ipynb)
   - [Ensemble Judges](Ensemble_Judges.ipynb)
   - Production Evaluation (this notebook)

2. **Implement for your use case:**
   - Start with single judge + sampling
   - Add monitoring and alerts
   - Iterate based on metrics

3. **Additional resources:**
   - [Testing Agents Guide](../../testing_agents.md)
   - [Agent TDD](../Agent_TDD/Function_Calling_Agent_TDD.ipynb)
   - [Function Calling Agent](../Function_Calling/Function_Calling_Agent.ipynb)